<a href="https://colab.research.google.com/github/ellenyifang2011/sea_level/blob/main/src/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sea Level Project**

In [1]:
import sys
import sklearn
from packaging import version

print("Welcome to the Ocean!")
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")
assert sys.version_info >= (3, 7)

from google.colab import drive
drive.mount('/content/drive')


Welcome to the Ocean!
Mounted at /content/drive


# Get the Data

In [2]:
import os, sys
import pandas as pd

def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def data_dir() -> str:
    if in_colab():
        # default Colab path after mounting Drive
        return "/content/drive/MyDrive/Data"
    return os.path.abspath("./data")  # local default

def path(*parts) -> str:
    return os.path.join(data_dir(), *parts)

def load_data():
    filename="sl_raw2.csv"
    return pd.read_csv(path(filename))

_rawdata = load_data()
_rawdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Date                            244 non-null    float64
 1   Sea Level Monthly Mean          244 non-null    float64
 2   Antarctic Ice Melt Rates(mass)  244 non-null    float64
 3   Greenland Ice Melt Rates(mass)  244 non-null    float64
 4   Global Temp Anomaly             244 non-null    float64
 5   El Nino                         244 non-null    float64
 6   AMOC                            196 non-null    float64
 7   CO2                             244 non-null    float64
dtypes: float64(8)
memory usage: 15.4 KB


## Take a Quick Look at the Data Structure

In [3]:
#Drop the Date Column
_rawdata = _rawdata.drop('Date', axis=1)
_rawdata = _rawdata.drop('AMOC', axis=1)
_rawdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Sea Level Monthly Mean          244 non-null    float64
 1   Antarctic Ice Melt Rates(mass)  244 non-null    float64
 2   Greenland Ice Melt Rates(mass)  244 non-null    float64
 3   Global Temp Anomaly             244 non-null    float64
 4   El Nino                         244 non-null    float64
 5   CO2                             244 non-null    float64
dtypes: float64(6)
memory usage: 11.6 KB



## Data preparations

In [4]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# 1. Dummy Time Series Data (e.g., 100 timesteps, 1 feature)
tf.random.set_seed(42)
traing_size = 0.8
size_train = int(len(_rawdata) * traing_size)
size_valid = int(0.5*(len(_rawdata) - size_train))
size_test = len(_rawdata) - size_train - size_valid
print(size_train, size_valid, size_test)
data_train = _rawdata.iloc[0:size_train]
data_valid = _rawdata.iloc[size_train:size_train+size_valid]
data_test = _rawdata.iloc[size_train+size_valid:]

# 2. Scale the data
# MUST fit on training data only to avoid leakage
scaler = StandardScaler()
scaled_train = scaler.fit_transform(data_train)
scaled_valid = scaler.fit_transform(data_valid)
scaled_test = scaler.fit_transform(data_test)

# 3. Create Windows
sequence_length = 36
half_sequence_length = 23
# X: windows of 6, y: next value
# Example: Using 6 past steps to predict the next single step
dataset_train = tf.keras.utils.timeseries_dataset_from_array(
    data = scaled_train,#scaled_train[:-1],
    targets = scaled_train[0:][sequence_length:],#scaled_train[1:],
    sequence_length = sequence_length,
    batch_size = 32,
    shuffle = False,
    seed=42
)

dataset_valid = tf.keras.utils.timeseries_dataset_from_array(
    scaled_valid,
    targets=scaled_valid[0:][half_sequence_length:],
    sequence_length=half_sequence_length,
    batch_size=32,
    shuffle = False
)

dataset_test = tf.keras.utils.timeseries_dataset_from_array(
    scaled_test,
    targets=scaled_test[0:][half_sequence_length:],
    sequence_length=half_sequence_length,
    batch_size=32,
    shuffle = False
)

195 24 25


## LSTM simple setup

In [ ]:
# 4. Define and Compile Model (using MAE)
tf.random.set_seed(42)
features = _rawdata.shape[-1]
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=(sequence_length, features)),
    tf.keras.layers.Dense(1)
])

# Use Mean Absolute Error (MAE) for regression
model.compile(optimizer='adam', loss='mae', metrics=['mae'])

# 5. Train
model.fit(dataset_train, validation_data = dataset_valid, epochs=75)

Epoch 1/75


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 115ms/step - loss: 0.7553 - mae: 0.7553 - val_loss: 1.3770 - val_mae: 1.3770
Epoch 2/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.7331 - mae: 0.7331 - val_loss: 1.3879 - val_mae: 1.3879
Epoch 3/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.7245 - mae: 0.7245 - val_loss: 1.3940 - val_mae: 1.3940
Epoch 4/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.7194 - mae: 0.7194 - val_loss: 1.3941 - val_mae: 1.3941
Epoch 5/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - loss: 0.7164 - mae: 0.7164 - val_loss: 1.3885 - val_mae: 1.3885
Epoch 6/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.7134 - mae: 0.7134 - val_loss: 1.3796 - val_mae: 1.3796
Epoch 7/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.7102 - mae: 0.7102 - val_loss: 1.3692 - val_mae: 1.3692
Epoch 8/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.7072 - mae: 0.7072 - val_loss: 1.3593 - val_mae: 1.3593
Epoch 9/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.7050 - mae: 0.7050 - val_loss:

In [ ]:
from sklearn.metrics import mean_absolute_error

# Make predictions on test set
y_pred = model.predict(dataset_test, verbose=0)

# Extract actual target values from the test dataset
# dataset_test yields batches of (inputs, targets)
# targets are scaled_test[sequence_length:] which means full rows of 6 features
all_actual_scaled_targets = []
for _, targets_batch in dataset_test:
    all_actual_scaled_targets.append(targets_batch.numpy())
concatenated_actual_scaled_targets = np.concatenate(all_actual_scaled_targets, axis=0)

# The model predicts a single feature (assuming 'Sea Level Monthly Mean', which is index 0)
actual_scaled_first_feature = concatenated_actual_scaled_targets[:, 0]

# Inverse transform y_pred (shape (19,1)) using the scaler's mean/std for the first feature
# y_pred needs to be flattened to (19,) for element-wise operation with scalar mean/std
y_pred_unscaled = y_pred.flatten() * scaler.scale_[0] + scaler.mean_[0]

# Inverse transform the actual first feature from the test set
actual_unscaled_first_feature = actual_scaled_first_feature * scaler.scale_[0] + scaler.mean_[0]

# Calculate the Mean Absolute Error (MAE)
mae = mean_absolute_error(actual_unscaled_first_feature, y_pred_unscaled)
#mae = np.round(mae, 5)

# Print the calculated MAE
print(f"Test MAE (manual prediction): {mae:.4f}")

Test MAE (manual prediction): 0.0438


## Candidate 1: LSTM Test MAE (manual prediction): 0.0143

In [7]:
import keras
from keras.callbacks import ReduceLROnPlateau
# 4. Define and Compile Model (using MAE)
tf.random.set_seed(42)
features = _rawdata.shape[-1]
model = tf.keras.Sequential([
    tf.keras.Input(shape=(sequence_length, features)),
    tf.keras.layers.LSTM(16, return_sequences =True, activation='relu'),
    tf.keras.layers.LSTM(16, activation='relu'),
    tf.keras.layers.Dense(16),
    tf.keras.layers.Dense(1)
])

# Define the optimizer with a specific learning rate
#optimizer = keras.optimizers.Adam(learning_rate=0.001) # Set the desired learning rate
optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # Set the desired learning rate

# Use Mean Absolute Error (MAE) for regression
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# 5. Train
callbacks = [
    # We use a callback to save the best-performing model. val_mae val_loss
    keras.callbacks.ModelCheckpoint("sl_model.keras", monitor='val_mae', save_best_only=True,mode='min',verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.0001)
]
model.fit(dataset_train, validation_data = dataset_valid, epochs=75, callbacks=callbacks)

Epoch 1/75
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.7892 - mae: 0.7242
Epoch 1: val_mae improved from inf to 1.48698, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 205ms/step - loss: 0.8194 - mae: 0.7368 - val_loss: 3.1193 - val_mae: 1.4870 - learning_rate: 0.0010
Epoch 2/75
3/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.7367 - mae: 0.7088
Epoch 2: val_mae did not improve from 1.48698
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.8035 - mae: 0.7292 - val_loss: 3.2151 - val_mae: 1.5130 - learning_rate: 0.0010
Epoch 3/75
3/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.7246 - mae: 0.7017
Epoch 3: val_mae did not improve from 1.48698
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.7936 - mae: 0.7243 - val_loss: 3.1389 - val_mae: 1.4924 - learning_rate: 0.0010
Epoch 4/75
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.7163 - mae: 0.6922
Epoch 4: val_mae did not improve from 1.48698
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.7904 - mae: 0.7228 - val_loss: 3.1905 

## Candidate 2: SimpleRNN

In [11]:
# 4. Define and Compile Model (using MAE)
tf.random.set_seed(42)
features = _rawdata.shape[-1]
model = tf.keras.Sequential([
    tf.keras.Input(shape=(sequence_length, features)),
    tf.keras.layers.SimpleRNN(16, return_sequences =True, activation='relu'),
    tf.keras.layers.SimpleRNN(16, activation='relu'),
    tf.keras.layers.Dense(16),
    tf.keras.layers.Dense(1)
])

# Define the optimizer with a specific learning rate
#optimizer = keras.optimizers.Adam(learning_rate=0.001) # Set the desired learning rate
optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # Set the desired learning rate

# Use Mean Absolute Error (MAE) for regression
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# 5. Train
# Pass the callback to the fit method
#reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
callbacks = [
    # We use a callback to save the best-performing model. val_mae val_loss
    keras.callbacks.ModelCheckpoint("sl_model.keras", monitor='val_mae', save_best_only=True,mode='min',verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.0001)
]
model.fit(dataset_train, validation_data = dataset_valid, epochs=100, callbacks=callbacks)

Epoch 1/100
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.7493 - mae: 0.7020
Epoch 1: val_mae improved from inf to 1.46756, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 144ms/step - loss: 0.8370 - mae: 0.7339 - val_loss: 3.0498 - val_mae: 1.4676 - learning_rate: 0.0010
Epoch 2/100
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.7070 - mae: 0.6854
Epoch 2: val_mae improved from 1.46756 to 1.36725, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.7915 - mae: 0.7191 - val_loss: 2.5994 - val_mae: 1.3673 - learning_rate: 0.0010
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.7432 - mae: 0.7022
Epoch 3: val_mae improved from 1.36725 to 1.32433, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.7765 - mae: 0.7156 - val_loss: 2.3663 - val_mae: 1.3243 - learning_rate: 0.0010
Epoch 4/100
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.6864 - mae: 0.6787
Epoch 4: val_mae improved from 1.32433 to 1.28654, saving

## Candidate 3: GRU

In [ ]:
# 4. Define and Compile Model (using MAE)
tf.random.set_seed(42)
features = _rawdata.shape[-1]
model = tf.keras.Sequential([
    tf.keras.Input(shape=(sequence_length, features)),
    tf.keras.layers.GRU(16, return_sequences =True, activation='relu'),
    tf.keras.layers.GRU(16, activation='relu'),
    tf.keras.layers.Dense(16),
    tf.keras.layers.Dense(1)
])

# Use Mean Absolute Error (MAE) for regression
#model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 5. Train
#model.fit(dataset_train, validation_data = dataset_valid, epochs=75)

# Define the optimizer with a specific learning rate
#optimizer = keras.optimizers.Adam(learning_rate=0.001) # Set the desired learning rate
optimizer = keras.optimizers.RMSprop(learning_rate=0.001) # Set the desired learning rate

# Use Mean Absolute Error (MAE) for regression
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# 5. Train
# Pass the callback to the fit method
#reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)
callbacks = [
    # We use a callback to save the best-performing model. val_mae val_loss
    keras.callbacks.ModelCheckpoint("sl_model.keras", monitor='val_mae', save_best_only=True,mode='min',verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.0001)
]
model.fit(dataset_train, validation_data = dataset_valid, epochs=100, callbacks=callbacks)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.8365 - mae: 0.7231
Epoch 1: val_mae improved from inf to 1.61454, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 197ms/step - loss: 0.8675 - mae: 0.7370 - val_loss: 3.6183 - val_mae: 1.6145 - learning_rate: 0.0010
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.7833 - mae: 0.7112
Epoch 2: val_mae improved from 1.61454 to 1.56945, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.8150 - mae: 0.7250 - val_loss: 3.4336 - val_mae: 1.5695 - learning_rate: 0.0010
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.7702 - mae: 0.7080
Epoch 3: val_mae improved from 1.56945 to 1.54427, saving model to sl_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.8016 - mae: 0.7215 - val_loss: 3.3344 - val_mae: 1.5443 - learning_rate: 0.0010
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.7619 - mae: 0.7057
Epoch 4: val_mae improved from 1.54427 to 1.52664, saving

In [12]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

model_best = keras.models.load_model("sl_model.keras")
# Make predictions on test set
y_pred = model_best.predict(dataset_test, verbose=0)

# Extract actual target values from the test dataset
# dataset_test yields batches of (inputs, targets)
# targets are scaled_test[sequence_length:] which means full rows of 6 features
all_actual_scaled_targets = []
for _, targets_batch in dataset_test:
    all_actual_scaled_targets.append(targets_batch.numpy())
concatenated_actual_scaled_targets = np.concatenate(all_actual_scaled_targets, axis=0)

# The model predicts a single feature (assuming 'Sea Level Monthly Mean', which is index 0)
actual_scaled_first_feature = concatenated_actual_scaled_targets[:, 0]

# Inverse transform y_pred (shape (19,1)) using the scaler's mean/std for the first feature
# y_pred needs to be flattened to (19,) for element-wise operation with scalar mean/std
y_pred_unscaled = y_pred.flatten() * scaler.scale_[0] + scaler.mean_[0]

# Inverse transform the actual first feature from the test set
actual_unscaled_first_feature = actual_scaled_first_feature * scaler.scale_[0] + scaler.mean_[0]

# Calculate the Mean Absolute Error (MAE)
mse = mean_squared_error(actual_unscaled_first_feature, y_pred_unscaled)
mae = mean_absolute_error(actual_unscaled_first_feature, y_pred_unscaled)
#mae = np.round(mae, 5)

# Print the calculated MAE
print(f"Test MSE (manual prediction): {mse:.6f}")
print(f"Test MAE (manual prediction): {mae:.6f}")

Test MSE (manual prediction): 0.000227
Test MAE (manual prediction): 0.014613
